# Lab 5 — Solución docente (RAG + Ollama)

Referencia con pipeline completo y respuestas teóricas sugeridas.
Los alumnos trabajan con `llm_local_estructuras_alumno_ia.ipynb` + bitácora de prompts.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_pasos_rag, verificar_ollama, verificar_inventario_pdfs,
    verificar_extraccion, verificar_chunks, verificar_embeddings,
    verificar_recuperacion, verificar_rag_respuesta, verificar_comparacion,
    listar_pdfs, extraer_texto_pdf, fragmentar_texto, cosine_top_k, llamar_ollama,
    resumen_seccion, E030_MAX_PAGINAS,
)

import numpy as np
import requests
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer

MODELO_EMB = 'all-MiniLM-L6-v2'
RUTA_PDFS = Path('pdfs')

print('✅ Entorno listo (Codespaces / labs/.venv).')
print('   Embeddings:', MODELO_EMB, '| Ollama: http://localhost:11434')


## Corpus — Normas estructurales peruanas

| PDF | Norma | Uso en obra |
|-----|-------|-------------|
| `Norma_E_020_CARGAS.pdf` | **E.020** | Cargas muertas, vivas, viento |
| `Norma_E_030_SISMO.pdf` | **E.030** | Diseño sismorresistente (muestra 30 pág.) |
| `Norma_E_050_SUELOS.pdf` | **E.050** | Estudios geotécnicos |

Pipeline: **extracción → chunks → embeddings (MiniLM) → top-k → Ollama**.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Contexto RAG en consultas de normativa

| Enfoque | Ventaja | Riesgo |
|---------|---------|--------|
| **Prompt vacío** | Rápido | Alucinaciones, cifras inventadas |
| **RAG local** | Respuesta anclada a PDFs de obra | Calidad del chunk y del modelo pequeño |

### 📘 Subpreguntas
- **1.a** ¿Por qué un ingeniero preferiría RAG local frente a ChatGPT con el PDF subido?
- **1.b** ¿Qué paso del pipeline evita que el LLM «invente» un artículo?


#### ✅ Respuesta sugerida

**1.a** Privacidad: los PDFs no salen del entorno; control de versión del modelo.
**1.b** La **recuperación** (top-k): el LLM solo ve fragmentos reales extraídos del PDF.


In [ ]:
PASOS_RAG = ["Carga de PDFs", "Fragmentación en chunks", "Embeddings con MiniLM", "Recuperación top-k", "Generación con Ollama"]
print("Pipeline RAG:")
for i, paso in enumerate(PASOS_RAG, 1):
    print(f"  {i}. {paso}")


In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `PASOS_RAG`
r = []
try:
    r = verificar_pasos_rag(PASOS_RAG)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — Contexto RAG', r)


## Pregunta 2 — Verificar Ollama (LLM local)

### 📘 Subpreguntas
- **2.a** ¿Qué ventaja tiene ejecutar el LLM en localhost:11434?
- **2.b** ¿Qué hacer si `OLLAMA_OK` es False?


#### ✅ Respuesta sugerida

**2.a** Datos de obra no salen del contenedor; costo predecible sin tokens en la nube.
**2.b** Ejecutar `bash labs/lab5/_ollama_setup.sh` y volver a probar `/api/tags`.


In [ ]:
# --- PRE-ESCRITO: diagnóstico Ollama ---
# Si falla, ejecuta en terminal: bash labs/lab5/_ollama_setup.sh
print("Comprueba que Ollama esté activo antes de las secciones 8–9.")


In [ ]:
MODELO_LLM = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434"
try:
    resp = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    OLLAMA_OK = resp.status_code == 200
    modelos = [m['name'] for m in resp.json().get('models', [])] if OLLAMA_OK else []
except Exception:
    OLLAMA_OK = False
    modelos = []
print(f"OLLAMA_OK={OLLAMA_OK} | Modelos: {modelos}")


In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `MODELO_LLM`, `OLLAMA_OK`
r = []
try:
    r = verificar_ollama(MODELO_LLM, OLLAMA_OK)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — Ollama', r)


## Pregunta 3 — Inventario de PDFs

### 📘 Subpreguntas
- **3.a** ¿Cuántos documentos normativos hay en `pdfs/`?
- **3.b** ¿Por qué empezamos con E.020 (cargas)?


#### ✅ Respuesta sugerida

**3.a** Tres PDFs: E.020, E.030, E.050.
**3.b** Es el más compacto y las consultas de cargas son frecuentes en diseño.


In [ ]:
# --- PRE-ESCRITO: listar corpus ---
lista_pdfs = listar_pdfs(RUTA_PDFS)
for p in lista_pdfs:
    print(f"  · {p.name} ({p.stat().st_size // 1024} KB)")


In [ ]:
N_PDFS = len(lista_pdfs)
PDF_ACTIVO = "Norma_E_020_CARGAS.pdf"
print(f"N_PDFS={N_PDFS} | PDF_ACTIVO={PDF_ACTIVO}")


In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `PDF_ACTIVO`, `N_PDFS`
r = []
try:
    r = verificar_inventario_pdfs(PDF_ACTIVO, N_PDFS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Inventario PDFs', r)


## Pregunta 4 — Extracción de texto (pypdf)

### 📘 Subpreguntas
- **4.a** ¿Por qué extraemos texto antes de preguntar al LLM?
- **4.b** ¿Qué limitación tienen los PDFs escaneados (solo imagen)?


#### ✅ Respuesta sugerida

**4.a** El LLM no lee PDF binario; necesita texto plano fragmentable.
**4.b** Sin OCR no hay texto extraíble — requerirían otra herramienta.


In [ ]:
# --- PRE-ESCRITO: ruta del PDF activo ---
ruta_activo = RUTA_PDFS / PDF_ACTIVO
print(f"Extrayendo: {ruta_activo}")


In [ ]:
texto_pdf, N_PAGINAS = extraer_texto_pdf(ruta_activo)
N_CARACTERES = len(texto_pdf)
print(f"Páginas: {N_PAGINAS} | Caracteres: {N_CARACTERES}")
print(texto_pdf[:400])


In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `texto_pdf`, `N_PAGINAS`, `N_CARACTERES`
r = []
try:
    r = verificar_extraccion(texto_pdf, N_PAGINAS, N_CARACTERES)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — Extracción', r)


## Pregunta 5 — Fragmentación en chunks

### 📘 Subpreguntas
- **5.a** ¿Por qué no enviamos el PDF completo al LLM?
- **5.b** ¿Para qué sirve `CHUNK_OVERLAP`?


#### ✅ Respuesta sugerida

**5.a** Límite de contexto del modelo; recuperación más precisa con trozos pequeños.
**5.b** Evita cortar definiciones o tablas justo en el límite del fragmento.


In [ ]:
# --- PRE-ESCRITO: parámetros sugeridos ---
# CHUNK_SIZE ≈ 600–1000 caracteres | CHUNK_OVERLAP ≈ 10–15 % del tamaño
print("Ajusta CHUNK_SIZE si obtienes demasiados o muy pocos fragmentos.")


In [ ]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
CHUNKS = fragmentar_texto(texto_pdf, CHUNK_SIZE, CHUNK_OVERLAP)
N_CHUNKS = len(CHUNKS)
print(f"N_CHUNKS={N_CHUNKS}")
print(CHUNKS[0][:200], "...")


In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `CHUNKS`, `N_CHUNKS`, `CHUNK_SIZE`, `CHUNK_OVERLAP`
r = []
try:
    r = verificar_chunks(CHUNKS, N_CHUNKS, CHUNK_SIZE, CHUNK_OVERLAP)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — Fragmentación', r)


## Pregunta 6 — Embeddings e índice en memoria

### 📘 Subpreguntas
- **6.a** ¿Qué representa cada fila de `EMBEDDINGS`?
- **6.b** ¿Por qué usamos MiniLM y reservamos Ollama solo para generación?


#### ✅ Respuesta sugerida

**6.a** Un vector numérico del significado de un chunk de normativa.
**6.b** MiniLM es ligero y estable en Codespaces; Ollama concentra RAM en el LLM.


In [ ]:
# --- PRE-ESCRITO: indexar E.050 y muestra E.030 ---
def _texto_norma(nombre: str, max_pag: int | None = None) -> str:
    t, _ = extraer_texto_pdf(RUTA_PDFS / nombre, max_paginas=max_pag)
    return t

texto_e050, _ = extraer_texto_pdf(RUTA_PDFS / 'Norma_E_050_SUELOS.pdf')
texto_e030, _ = extraer_texto_pdf(RUTA_PDFS / 'Norma_E_030_SISMO.pdf', max_paginas=E030_MAX_PAGINAS)
texto_corpus = texto_pdf + '\n' + texto_e050 + '\n' + texto_e030
CHUNKS_CORPUS = fragmentar_texto(texto_corpus, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Corpus ampliado: {len(CHUNKS_CORPUS)} chunks (E.020 + E.050 + E.030 parcial)")


In [ ]:
modelo_emb = SentenceTransformer(MODELO_EMB)
CHUNKS = CHUNKS_CORPUS
N_CHUNKS = len(CHUNKS)
EMBEDDINGS = modelo_emb.encode(CHUNKS, show_progress_bar=False)
N_VECTORES = EMBEDDINGS.shape[0]
print(f"N_VECTORES={N_VECTORES} | shape={EMBEDDINGS.shape}")


In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `EMBEDDINGS`, `N_VECTORES`
r = []
try:
    r = verificar_embeddings(EMBEDDINGS, N_VECTORES, N_CHUNKS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Embeddings', r)


## Pregunta 7 — Recuperación top-k

### 📘 Subpreguntas
- **7.a** ¿Qué mide la similitud coseno entre consulta y chunk?
- **7.b** ¿Qué pasa si TOP_K es demasiado grande?


#### ✅ Respuesta sugerida

**7.a** Qué tan alineado está el significado de la pregunta con el fragmento.
**7.b** Se introduce ruido en el prompt; más tokens y posible confusión del LLM.


In [ ]:
# --- PRE-ESCRITO: visualizar función de recuperación ---
display(Markdown(
    "**Recuperación:** embed(query) → similitud coseno con cada fila de EMBEDDINGS → top-k chunks."
))


In [ ]:
CONSULTA = "¿Qué tipos de sobrecarga considera la norma E.020 de cargas?"
TOP_K = 3
q_vec = modelo_emb.encode([CONSULTA])[0]
idx, scores = cosine_top_k(q_vec, EMBEDDINGS, TOP_K)
CHUNKS_RECUPERADOS = [CHUNKS[i] for i in idx]
for i, (c, s) in enumerate(zip(CHUNKS_RECUPERADOS, scores), 1):
    print(f"--- Fragmento {i} (score={s:.3f}) ---\n{c[:300]}...\n")


In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `CONSULTA`, `CHUNKS_RECUPERADOS`, `TOP_K`
r = []
try:
    r = verificar_recuperacion(CONSULTA, CHUNKS_RECUPERADOS, TOP_K)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Recuperación', r)


## Pregunta 8 — Prompt RAG y respuesta con Ollama

### 📘 Subpreguntas
- **8.a** ¿Qué instrucción reduce alucinaciones en el prompt?
- **8.b** ¿Por qué mostramos los fragmentos recuperados antes de la respuesta?


#### ✅ Respuesta sugerida

**8.a** «Responde SOLO con base en el CONTEXTO» + admitir desconocimiento.
**8.b** Validación visual: el ingeniero contrasta respuesta vs fuente real.


In [ ]:
# --- PRE-ESCRITO: plantilla de system prompt ---
PLANTILLA_SISTEMA = """Responde SOLO con base en el CONTEXTO.
Si la respuesta no está en el contexto, di "No consta en el contexto".
Cita artículo o sección cuando sea posible."""
print(PLANTILLA_SISTEMA)


In [ ]:
contexto = "\n---\n".join(CHUNKS_RECUPERADOS)
PROMPT_RAG = f"""Eres un asistente de normativa estructural peruana.
Responde SOLO con base en el CONTEXTO. Si no está, di 'No consta en el contexto'.

CONTEXTO:
{contexto}

PREGUNTA: {CONSULTA}
"""
RESPUESTA_RAG = llamar_ollama(PROMPT_RAG, MODELO_LLM, OLLAMA_URL)
print("RESPUESTA RAG:\n", RESPUESTA_RAG)


In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `PROMPT_RAG`, `RESPUESTA_RAG`
r = []
try:
    r = verificar_rag_respuesta(PROMPT_RAG, RESPUESTA_RAG)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — Prompt RAG', r)


## Pregunta 9 — RAG vs consulta sin contexto

### 📘 Subpreguntas
- **9.a** ¿Qué diferencias observas entre ambas respuestas?
- **9.b** ¿Cuándo **no** deberías confiar en la salida del LLM?


#### ✅ Respuesta sugerida

**9.a** Sin RAG suele ser genérica; con RAG cita fragmentos concretos (si el modelo obedece).
**9.b** Decisiones de diseño estructural, cifras numéricas sin verificar en norma oficial.


In [ ]:
# --- PRE-ESCRITO: misma consulta, sin chunks ---
print(f"Consulta base: {CONSULTA}")
print("Generaremos RESPUESTA_SIN_RAG sin contexto recuperado.")


In [ ]:
prompt_vacio = f"Responde brevemente: {CONSULTA}"
RESPUESTA_SIN_RAG = llamar_ollama(prompt_vacio, MODELO_LLM, OLLAMA_URL)
print("=== CON RAG ===\n", RESPUESTA_RAG[:600])
print("\n=== SIN RAG ===\n", RESPUESTA_SIN_RAG[:600])
USA_FUENTE = any(p in RESPUESTA_RAG.lower() for p in ["e.020", "norma", "artículo", "sobrecarga", "carga"])
print(f"\nUSA_FUENTE={USA_FUENTE}")


In [ ]:
# --- Autoevaluación 9 ---
# Requiere (celda «Aquí coloca tu solución»): `RESPUESTA_SIN_RAG`, `USA_FUENTE`
r = []
try:
    r = verificar_comparacion(RESPUESTA_SIN_RAG, USA_FUENTE, RESPUESTA_RAG)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('9 — Comparación RAG', r)


## Cierre docente

- Contrasta siempre fragmentos recuperados vs respuesta.
- E.030 completo requiere más RAM; en producción usar índice persistente.
- Bitácora de prompts: ver `prompts_entregados.md`.
